In [313]:
import pandas as pd
import re
import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import stopwords
from spellchecker import SpellChecker
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import GridSearchCV
from gensim.models import Word2Vec
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

In [58]:
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\dmitr\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\dmitr\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\dmitr\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
pos = pd.read_csv('data/processedPositive.csv').T.reset_index().rename(columns={'index':'tweets'})
net = pd.read_csv('data/processedNeutral.csv').T.reset_index().rename(columns={'index':'tweets'})
neg = pd.read_csv('data/processedNegative.csv').T.reset_index().rename(columns={'index':'tweets'})

In [5]:
df_lst = [pos, net, neg]
for i, df in enumerate(df_lst):
    df['label'] = 1 - i

full_df = pd.concat(df_lst, ignore_index=True)

**Виды обработки текста:**
1) tokenization - Процесс разбиения текста на отдельные токены
2) stemming - Приведение слов к основе путём механического отсечения окончаний без учёта контекста и грамматики
3) lemmatization - Приведение слова к его словарной (нормальной) форме с учётом морфологии и части речи
4) stemming+misspelling
5) lemmatization+misspelling
   
Misspellings - Исправление орфографических ошибок
   
**Виды векторизации:**
1) bag of words
2) word counts
3) TFIDF
4) word2vec(бонус)

### 1. Preprocessing

Готовим датасеты для обучения

Виды обработки текста:

1. tokenization
2. stemming
3. lemmatization
4. stemming+misspelling
5. lemmatization+misspelling
   
Виды векторизации: 
1. bag of words
2. word counts
3. TFIDF
4. word2vec(бонус)

In [12]:
full_df

,tweets,label
0,An inspiration in all aspects: Fashion,1
1,fitness,1
2,beauty and personality. :)KISSES TheFashionIcon,1
3,Apka Apna Awam Ka Channel Frankline Tv Aam Adm...,1
4,Beautiful album from the greatest unsung guit...,1
...,...,...
3868,wish knock out lang talaga for the new school ...,-1
3869,i miss so much unhappy,-1
3870,Same unhappy .1,-1
3871,Hi instant message your friend friend lang,-1


In [64]:
def prep_txt(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text) # удаляем пунктуацию
    text = re.sub(r"\d+", " ", text) # удаляем цифры
    text = re.sub(r"\s+", " ", text).strip() # удаляем пробелы
    return text.split()

In [70]:
def remove_sw(tokens):
    stop_words = stopwords.words('english')
    tokens = [t for t in tokens if t not in stop_words]
    return tokens

In [72]:
def stemming(tokens):
    ps = PorterStemmer()
    tokens = [ps.stem(t) for t in tokens]
    return tokens

In [81]:
def lems(tokens):
    wl = WordNetLemmatizer()
    tokens = [wl.lemmatize(t) for t in tokens]
    return tokens

In [83]:
def fix_misspellings(tokens):
    spell = SpellChecker()
    fixed_tokens = []
    for token in tokens:
        if token.isdigit():
            fixed_tokens.append(token)
            continue
        corrected = spell.correction(token)
        if corrected is None:
            fixed_tokens.append(token)
        else:
            fixed_tokens.append(corrected)
    return fixed_tokens

In [147]:
%%time
full_df['tokens'] = full_df['tweets'].apply(prep_txt)
full_df['tokens_not_stopwords'] = full_df['tokens'].apply(remove_sw)

CPU times: total: 734 ms
Wall time: 756 ms


In [148]:
%%time
full_df['stemming'] = full_df['tokens_not_stopwords'].apply(stemming)
full_df['stemming_misspillings'] = full_df['tokens_not_stopwords'].apply(fix_misspellings).apply(stemming)

CPU times: total: 12min 54s
Wall time: 12min 57s


In [150]:
%%time
full_df['lemmatization'] = full_df['tokens_not_stopwords'].apply(lems)
full_df['lemmatization_misspillings'] = full_df['tokens_not_stopwords'].apply(fix_misspellings).apply(lems)

CPU times: total: 13min 34s
Wall time: 13min 36s


In [151]:
full_df.head()

,tweets,label,tokens,tokens_not_stopwords,stemming,stemming_misspillings,lemmatization,lemmatization_misspillings
0,An inspiration in all aspects: Fashion,1,"[an, inspiration, in, all, aspects, fashion]","[inspiration, aspects, fashion]","[inspir, aspect, fashion]","[inspir, aspect, fashion]","[inspiration, aspect, fashion]","[inspiration, aspect, fashion]"
1,fitness,1,[fitness],[fitness],[fit],[fit],[fitness],[fitness]
2,beauty and personality. :)KISSES TheFashionIcon,1,"[beauty, and, personality, kisses, thefashioni...","[beauty, personality, kisses, thefashionicon]","[beauti, person, kiss, thefashionicon]","[beauti, person, kiss, thefashionicon]","[beauty, personality, kiss, thefashionicon]","[beauty, personality, kiss, thefashionicon]"
3,Apka Apna Awam Ka Channel Frankline Tv Aam Adm...,1,"[apka, apna, awam, ka, channel, frankline, tv,...","[apka, apna, awam, ka, channel, frankline, tv,...","[apka, apna, awam, ka, channel, franklin, tv, ...","[aka, anna, away, ka, channel, franklin, to, a...","[apka, apna, awam, ka, channel, frankline, tv,...","[aka, anna, away, ka, channel, franklin, to, a..."
4,Beautiful album from the greatest unsung guit...,1,"[beautiful, album, from, the, greatest, unsung...","[beautiful, album, greatest, unsung, guitar, g...","[beauti, album, greatest, unsung, guitar, geni...","[beauti, album, greatest, unsung, guitar, geni...","[beautiful, album, greatest, unsung, guitar, g...","[beautiful, album, greatest, unsung, guitar, g..."


In [93]:
full_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3873 entries, 0 to 3872
Data columns (total 8 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   tweets                      3873 non-null   object
 1   label                       3873 non-null   int64 
 2   tokens                      3873 non-null   object
 3   tokens_not_stopwords        3873 non-null   object
 4   stemming                    3873 non-null   object
 5   stemming_misspillings       3873 non-null   object
 6   lemmatization               3873 non-null   object
 7   lemmatization_misspillings  3873 non-null   object
dtypes: int64(1), object(7)
memory usage: 242.2+ KB


In [95]:
full_df.to_csv('data/prepared_df.csv', index=False)

In [152]:
def check_emotion(text_tokens):
    text = text_tokens[0]
    tokens = text_tokens[1]
    if text.count(')') > text.count('('):
        tokens.append(')')
    elif text.count(')') < text.count('('):
        tokens.append('(')
    return tokens

In [153]:
%%time
for col in full_df.columns[2:]:
    full_df[col] = full_df[['tweets', col]].apply(check_emotion, axis=1)

C:\Users\dmitr\AppData\Local\Temp\ipykernel_25516\2376440936.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  text = text_tokens[0]
C:\Users\dmitr\AppData\Local\Temp\ipykernel_25516\2376440936.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  tokens = text_tokens[1]
C:\Users\dmitr\AppData\Local\Temp\ipykernel_25516\2376440936.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  text = text_tokens[0]
C:\Users\dmitr\AppData\Local\

CPU times: total: 500 ms
Wall time: 507 ms


C:\Users\dmitr\AppData\Local\Temp\ipykernel_25516\2376440936.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  text = text_tokens[0]
C:\Users\dmitr\AppData\Local\Temp\ipykernel_25516\2376440936.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  tokens = text_tokens[1]
C:\Users\dmitr\AppData\Local\Temp\ipykernel_25516\2376440936.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  text = text_tokens[0]
C:\Users\dmitr\AppData\Local\

In [154]:
full_df.head()

,tweets,label,tokens,tokens_not_stopwords,stemming,stemming_misspillings,lemmatization,lemmatization_misspillings
0,An inspiration in all aspects: Fashion,1,"[an, inspiration, in, all, aspects, fashion]","[inspiration, aspects, fashion]","[inspir, aspect, fashion]","[inspir, aspect, fashion]","[inspiration, aspect, fashion]","[inspiration, aspect, fashion]"
1,fitness,1,[fitness],[fitness],[fit],[fit],[fitness],[fitness]
2,beauty and personality. :)KISSES TheFashionIcon,1,"[beauty, and, personality, kisses, thefashioni...","[beauty, personality, kisses, thefashionicon, )]","[beauti, person, kiss, thefashionicon, )]","[beauti, person, kiss, thefashionicon, )]","[beauty, personality, kiss, thefashionicon, )]","[beauty, personality, kiss, thefashionicon, )]"
3,Apka Apna Awam Ka Channel Frankline Tv Aam Adm...,1,"[apka, apna, awam, ka, channel, frankline, tv,...","[apka, apna, awam, ka, channel, frankline, tv,...","[apka, apna, awam, ka, channel, franklin, tv, ...","[aka, anna, away, ka, channel, franklin, to, a...","[apka, apna, awam, ka, channel, frankline, tv,...","[aka, anna, away, ka, channel, franklin, to, a..."
4,Beautiful album from the greatest unsung guit...,1,"[beautiful, album, from, the, greatest, unsung...","[beautiful, album, greatest, unsung, guitar, g...","[beauti, album, greatest, unsung, guitar, geni...","[beauti, album, greatest, unsung, guitar, geni...","[beautiful, album, greatest, unsung, guitar, g...","[beautiful, album, greatest, unsung, guitar, g..."


In [161]:
full_df.to_csv('data/prepared_df2.csv', index=False)

### 2. Train test split & vectorizing

In [232]:
X = full_df.drop(['label'], axis=1)
y = full_df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

In [214]:
def vectorize(X_train, X_test, vectorizer_column, mode):
    binary_vectorizer = CountVectorizer(binary=True)
    count_vectorizer = CountVectorizer()
    tfidf_vectorizer = TfidfVectorizer()

    X_train_binary = binary_vectorizer.fit_transform(X_train[vectorizer_column].apply(lambda x: ' '.join(x)))
    X_test_binary = binary_vectorizer.transform(X_test[vectorizer_column].apply(lambda x: ' '.join(x)))

    X_train_count = count_vectorizer.fit_transform(X_train[vectorizer_column].apply(lambda x: ' '.join(x)))
    X_test_count = count_vectorizer.transform(X_test[vectorizer_column].apply(lambda x: ' '.join(x)))

    X_train_tfidf = tfidf_vectorizer.fit_transform(X_train[vectorizer_column].apply(lambda x: ' '.join(x)))
    X_test_tfidf = tfidf_vectorizer.transform(X_test[vectorizer_column].apply(lambda x: ' '.join(x)))

    if mode == 'word counts':
        return X_train_count, X_test_count
    elif mode == 'bag of words':
        return X_train_binary, X_test_binary
    elif mode == 'TFIDF':
        return X_train_tfidf, X_test_tfidf

### 3. Pivot accuracy

In [238]:
rows = X_train.columns[1:]
cols = [
    "bag of words",
    "word counts",
    "TFIDF",
    "word2vec"
]

pivot_result = pd.DataFrame(index=rows, columns=cols)
pivot_result

,bag of words,word counts,TFIDF,word2vec
tokens,NaN,NaN,NaN,NaN
tokens_not_stopwords,NaN,NaN,NaN,NaN
stemming,NaN,NaN,NaN,NaN
stemming_misspillings,NaN,NaN,NaN,NaN
lemmatization,NaN,NaN,NaN,NaN
lemmatization_misspillings,NaN,NaN,NaN,NaN


In [304]:
def gs_params(X_train, y_train, name):
    param_grid = {
        'C': [0.1, 1, 10],
        'penalty': ['l2'],
        'solver': ['lbfgs', 'saga'],
        'max_iter': [100, 200],
        'class_weight': [None, 'balanced']
    }
    model = LogisticRegression()
    grid = GridSearchCV(model, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train, y_train)
    print(name)
    print(f'Best params: {grid.best_params_}\n')
    return grid.best_estimator_

In [306]:
clf = LogisticRegression()

prepared_cols = X_train.columns[1:]
vec_df_idx = pivot_result.columns[:-1]

train_matrix = {}

def make_accuracy(X_train, X_test, y_train, y_test, name):
    clf = gs_params(X_train, y_train, name)
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    return accuracy_score(y_test, pred)

for col1 in prepared_cols:
    for col2 in vec_df_idx:
        X_train_vec, X_test_vec = vectorize(X_train, X_test, col1, col2)
        name = col1 + ' ' + col2
        train_matrix[name] = X_train_vec
        accuracy = make_accuracy(X_train_vec, X_test_vec, y_train, y_test, name)
        pivot_result.loc[col1, col2] = round(accuracy, 4)

tokens bag of words
Best params: {'C': 1, 'class_weight': 'balanced', 'max_iter': 100, 'penalty': 'l2', 'solver': 'lbfgs'}

tokens word counts
Best params: {'C': 1, 'class_weight': 'balanced', 'max_iter': 100, 'penalty': 'l2', 'solver': 'lbfgs'}

tokens TFIDF
Best params: {'C': 10, 'class_weight': 'balanced', 'max_iter': 100, 'penalty': 'l2', 'solver': 'lbfgs'}

tokens_not_stopwords bag of words
Best params: {'C': 1, 'class_weight': None, 'max_iter': 100, 'penalty': 'l2', 'solver': 'saga'}

tokens_not_stopwords word counts
Best params: {'C': 1, 'class_weight': 'balanced', 'max_iter': 100, 'penalty': 'l2', 'solver': 'lbfgs'}

tokens_not_stopwords TFIDF
Best params: {'C': 10, 'class_weight': 'balanced', 'max_iter': 100, 'penalty': 'l2', 'solver': 'saga'}

stemming bag of words
Best params: {'C': 1, 'class_weight': 'balanced', 'max_iter': 100, 'penalty': 'l2', 'solver': 'lbfgs'}

stemming word counts
Best params: {'C': 1, 'class_weight': 'balanced', 'max_iter': 100, 'penalty': 'l2', 'solv

In [247]:
pivot_result # без GridSearchCV

,bag of words,word counts,TFIDF,word2vec
tokens,0.889,0.889,0.8929,NaN
tokens_not_stopwords,0.889,0.889,0.8852,NaN
stemming,0.8865,0.8865,0.8903,NaN
stemming_misspillings,0.8852,0.8865,0.8916,NaN
lemmatization,0.8839,0.8826,0.8826,NaN
lemmatization_misspillings,0.8826,0.8826,0.8852,NaN


In [308]:
pivot_result # после GridSearchCV

,bag of words,word counts,TFIDF,word2vec
tokens,0.8903,0.8916,0.8981,NaN
tokens_not_stopwords,0.889,0.8994,0.8929,NaN
stemming,0.8916,0.8929,0.8877,NaN
stemming_misspillings,0.8903,0.889,0.8839,NaN
lemmatization,0.8852,0.889,0.8877,NaN
lemmatization_misspillings,0.8826,0.8865,0.8813,NaN


### 10 похожих твитов

In [271]:
def top_k_similar_pairs(sparse_matrix,k = 10):
    sim_matrix = cosine_similarity(sparse_matrix, dense_output=False)

    n = sparse_matrix.shape[0]
    sim = sim_matrix.toarray()

    np.fill_diagonal(sim, -np.inf)
    flat_idx = np.argsort(-sim, axis=None)
    pairs = []
    seen = set()
    for idx in flat_idx:
        i, j = np.unravel_index(idx, sim.shape)
        if i == j:
            continue
        a, b = (i, j) if i < j else (j, i)
        if (a, b) in seen:
            continue
        seen.add((a, b))
        pairs.append((a, b, float(sim[i, j])))
        if len(pairs) >= k:
            break
    return pairs

In [293]:
n = 1
for k, matrix in train_matrix.items():
    print(f'Датасет № {n}')
    print(k)
    print()
    pairs = top_k_similar_pairs(matrix,k = 10)
    for i, p in enumerate(pairs):
        print(f'Пара твитов № {i + 1}:')
        print(X_train.iloc[p[0]]['tweets'])
        print(X_train.iloc[p[1]]['tweets'])
        print()
    n += 1

Датасет № 1
tokens bag of words

Пара твитов № 1:
talking to my over driver about where I'm goinghe said he'd love to go to New York too but since Trump it's probably not.2
talking to my over driver about where I'm goinghe said he'd love to go to New York too but since Trump it's probably not.5

Пара твитов № 2:
talking to my over driver about where I'm goinghe said he'd love to go to New York too but since Trump it's probably not.5
talking to my over driver about where I'm goinghe said he'd love to go to New York too but since Trump it's probably not.8

Пара твитов № 3:
talking to my over driver about where I'm goinghe said he'd love to go to New York too but since Trump it's probably not.5
talking to my over driver about where I'm goinghe said he'd love to go to New York too but since Trump it's probably not.6

Пара твитов № 4:
talking to my over driver about where I'm goinghe said he'd love to go to New York too but since Trump it's probably not.7
talking to my over driver about whe

### W2V

In [343]:
class Word2VecVectorizer(BaseEstimator, TransformerMixin):
    def __init__(self, w2v_model, vector_size=256):
        self.w2v_model = w2v_model
        self.vector_size = vector_size

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        vectors = []
        for tokens in X:
            vec_list = []
            for t in tokens:
                if t in self.w2v_model.wv:
                    vec_list.append(self.w2v_model.wv[t])
            if len(vec_list) == 0:
                vec = np.zeros(self.vector_size)
            else:
                vec = np.mean(vec_list, axis=0)
            vectors.append(vec)
        return np.vstack(vectors)

In [395]:
for col in X_train.columns[1:]:
    X_raw = X_train[col].tolist()
    vector_size = 256
    w2v_model = Word2Vec(
        sentences=X_raw,
        vector_size=vector_size,
        window=4,
        min_count=2,
        workers=8,
        sg=4,
        epochs=60
    )
    
    
    vectorizer = Word2VecVectorizer(w2v_model=w2v_model, vector_size=vector_size)
    scaler = StandardScaler()
    X_w2v = vectorizer.fit_transform(X_raw)
    X_w2v = scaler.fit_transform(X_w2v)
    X_w2v_test = vectorizer.transform(X_test[col].tolist())
    X_w2v_test = scaler.transform(X_w2v_test)

    clf = LogisticRegression(solver="saga", class_weight="balanced", max_iter=2000, random_state=21)
    clf.fit(X_w2v, y_train)
    pred = clf.predict(X_w2v_test)
    result = accuracy_score(y_test, pred)
    pivot_result.loc[col, 'word2vec'] = round(result, 4)

In [396]:
pivot_result

,bag of words,word counts,TFIDF,word2vec
tokens,0.8903,0.8916,0.8981,0.8865
tokens_not_stopwords,0.889,0.8994,0.8929,0.8748
stemming,0.8916,0.8929,0.8877,0.8606
stemming_misspillings,0.8903,0.889,0.8839,0.8735
lemmatization,0.8852,0.889,0.8877,0.8774
lemmatization_misspillings,0.8826,0.8865,0.8813,0.8761
